# Estudo de Simulacao em MRLS (Notebook Unico para Colab)

Este notebook consolida toda a entrega em um unico arquivo: simulacao, metricas, graficos e interpretacao.

**Cenarios incluidos:** classico normal, caudas pesadas, erros assimetricos, media do erro nao nula (constante e dependente de X), erros AR(1) e relacao exponencial com comparacao entre escala original e transformacao log.

In [6]:
# Se estiver no Colab, execute esta celula para instalar dependencias
# !pip -q install numpy pandas scipy statsmodels matplotlib

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

@dataclass(frozen=True)
class StudyConfig:
    seed: int = 20260507
    replications: int = 1000
    sample_sizes: tuple = (30, 100)
    alpha: float = 0.05
    beta0: float = 1.0
    beta1: float = 2.0
    sigma: float = 1.0
    x_low: float = -2.0
    x_high: float = 2.0
    t_df: int = 3
    gamma_shape: float = 2.0
    gamma_scale: float = 1.0
    delta_const: float = 1.0
    delta_x: float = 0.6
    ar1_rho: float = 0.7
    lognormal_sigma: float = 0.4

SCENARIO_LABELS = {
    'classic_normal': 'C1_classico_normal',
    'heavy_tails': 'C2_caudas_pesadas',
    'skewed_errors': 'C3_erros_assimetricos',
    'nonzero_mean_const': 'C4a_media_erro_nao_nula_constante',
    'nonzero_mean_x_dep': 'C4b_media_erro_dependente_X',
    'ar1_errors': 'C5_erros_correlacionados_ar1',
    'exponential_lognormal': 'C6_relacao_exponencial_lognormal',
}

SCENARIOS = list(SCENARIO_LABELS.keys())
cfg = StudyConfig()

In [8]:
def generate_x(n, rng, cfg):
    return rng.uniform(cfg.x_low, cfg.x_high, size=n)

def _normal_errors(n, rng, sigma):
    return rng.normal(0.0, sigma, size=n)

def _heavy_tail_errors(n, rng, cfg):
    base = rng.standard_t(df=cfg.t_df, size=n)
    scale = cfg.sigma / np.sqrt(cfg.t_df / (cfg.t_df - 2))
    return base * scale

def _skewed_errors(n, rng, cfg):
    g = rng.gamma(shape=cfg.gamma_shape, scale=cfg.gamma_scale, size=n)
    g_centered = g - (cfg.gamma_shape * cfg.gamma_scale)
    g_sd = np.sqrt(cfg.gamma_shape * (cfg.gamma_scale**2))
    return (g_centered / g_sd) * cfg.sigma

def _ar1_errors(n, rng, cfg):
    rho = cfg.ar1_rho
    innovations_sd = cfg.sigma * np.sqrt(1 - rho**2)
    u = rng.normal(0.0, innovations_sd, size=n)
    eps = np.zeros(n)
    eps[0] = rng.normal(0.0, cfg.sigma)
    for i in range(1, n):
        eps[i] = rho * eps[i - 1] + u[i]
    return eps

def generate_dataset(scenario, n, rng, cfg, beta0, beta1):
    x = generate_x(n, rng, cfg)
    if scenario == 'classic_normal':
        eps = _normal_errors(n, rng, cfg.sigma)
        y = beta0 + beta1 * x + eps
    elif scenario == 'heavy_tails':
        eps = _heavy_tail_errors(n, rng, cfg)
        y = beta0 + beta1 * x + eps
    elif scenario == 'skewed_errors':
        eps = _skewed_errors(n, rng, cfg)
        y = beta0 + beta1 * x + eps
    elif scenario == 'nonzero_mean_const':
        eps = cfg.delta_const + _normal_errors(n, rng, cfg.sigma)
        y = beta0 + beta1 * x + eps
    elif scenario == 'nonzero_mean_x_dep':
        eps = cfg.delta_x * x + _normal_errors(n, rng, cfg.sigma)
        y = beta0 + beta1 * x + eps
    elif scenario == 'ar1_errors':
        eps = _ar1_errors(n, rng, cfg)
        y = beta0 + beta1 * x + eps
    elif scenario == 'exponential_lognormal':
        eps = rng.normal(0.0, cfg.lognormal_sigma, size=n)
        eta = np.exp(eps)
        y = np.exp(beta0 + beta1 * x) * eta
    else:
        raise ValueError('Cenario invalido')
    return x, y

def fit_ols(x, y, alpha):
    X = sm.add_constant(x, has_constant='add')
    model = sm.OLS(y, X).fit()
    ci = model.conf_int(alpha=alpha)
    return {
        'beta0_hat': float(model.params[0]),
        'beta1_hat': float(model.params[1]),
        'ci_beta0_low': float(ci[0, 0]),
        'ci_beta0_high': float(ci[0, 1]),
        'ci_beta1_low': float(ci[1, 0]),
        'ci_beta1_high': float(ci[1, 1]),
        'pvalue_beta1': float(model.pvalues[1]),
        'fitted': np.asarray(model.fittedvalues),
        'resid': np.asarray(model.resid),
    }

def summarize_parameter(estimates, true_value):
    mean_hat = float(np.mean(estimates))
    bias = mean_hat - true_value
    variance = float(np.var(estimates, ddof=1))
    mse = float(np.mean((estimates - true_value) ** 2))
    return mean_hat, bias, variance, mse

def coverage_rate(ci_low, ci_high, true_value):
    return float(np.mean((ci_low <= true_value) & (true_value <= ci_high)))

def rejection_rate(pvalues, alpha):
    return float(np.mean(pvalues < alpha))

def fit_for_scenario(scenario, x, y, alpha, model_label):
    if scenario == 'exponential_lognormal' and model_label == 'log_transform':
        return fit_ols(x, np.log(y), alpha)
    return fit_ols(x, y, alpha)

def collect_replications(scenario, n, cfg, rng, beta1_generation, model_label):
    rows = []
    diagnostic = None
    for _ in range(cfg.replications):
        x, y = generate_dataset(scenario, n, rng, cfg, cfg.beta0, beta1_generation)
        fit = fit_for_scenario(scenario, x, y, cfg.alpha, model_label)
        rows.append({
            'beta0_hat': fit['beta0_hat'],
            'beta1_hat': fit['beta1_hat'],
            'ci_beta0_low': fit['ci_beta0_low'],
            'ci_beta0_high': fit['ci_beta0_high'],
            'ci_beta1_low': fit['ci_beta1_low'],
            'ci_beta1_high': fit['ci_beta1_high'],
            'pvalue_beta1': fit['pvalue_beta1'],
        })
        if diagnostic is None:
            diagnostic = {'fitted': fit['fitted'], 'resid': fit['resid']}
    return pd.DataFrame(rows), diagnostic

def save_diagnostics(fig_dir, slug, diagnostic, beta1_values):
    plt.figure(figsize=(7,4))
    plt.scatter(diagnostic['fitted'], diagnostic['resid'], alpha=0.6)
    plt.axhline(0, color='black', linestyle='--')
    plt.title(f'Residuos vs ajustados - {slug}')
    plt.tight_layout(); plt.savefig(fig_dir / f'{slug}_residuos_vs_ajustados.png', dpi=150); plt.close()

    plt.figure(figsize=(6,6))
    stats.probplot(diagnostic['resid'], dist='norm', plot=plt)
    plt.title(f'QQ-plot residuos - {slug}')
    plt.tight_layout(); plt.savefig(fig_dir / f'{slug}_qqplot_residuos.png', dpi=150); plt.close()

    plt.figure(figsize=(7,4))
    plt.hist(beta1_values, bins=30, edgecolor='black', alpha=0.85)
    plt.title(f'Histograma beta1 - {slug}')
    plt.tight_layout(); plt.savefig(fig_dir / f'{slug}_hist_beta1.png', dpi=150); plt.close()

    plt.figure(figsize=(6,4))
    plt.boxplot(beta1_values, vert=True)
    plt.title(f'Boxplot beta1 - {slug}')
    plt.tight_layout(); plt.savefig(fig_dir / f'{slug}_boxplot_beta1.png', dpi=150); plt.close()

In [9]:
ROOT = Path.cwd().parent
OUTPUTS = ROOT / 'outputs'
TABLES = OUTPUTS / 'tables'
FIGURES = OUTPUTS / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(cfg.seed)
summary_rows = []
estimation_rows = []
representative_n = max(cfg.sample_sizes)

for scenario in SCENARIOS:
    model_labels = ['original_scale']
    if scenario == 'exponential_lognormal':
        model_labels = ['original_scale', 'log_transform']
    for n in cfg.sample_sizes:
        for model_label in model_labels:
            alt, diagnostic = collect_replications(scenario, n, cfg, rng, cfg.beta1, model_label)
            null, _ = collect_replications(scenario, n, cfg, rng, 0.0, model_label)
            size_h0 = rejection_rate(null['pvalue_beta1'].to_numpy(), cfg.alpha)
            power_h1 = rejection_rate(alt['pvalue_beta1'].to_numpy(), cfg.alpha)

            m0, b0, v0, mse0 = summarize_parameter(alt['beta0_hat'].to_numpy(), cfg.beta0)
            m1, b1, v1, mse1 = summarize_parameter(alt['beta1_hat'].to_numpy(), cfg.beta1)
            cov0 = coverage_rate(alt['ci_beta0_low'].to_numpy(), alt['ci_beta0_high'].to_numpy(), cfg.beta0)
            cov1 = coverage_rate(alt['ci_beta1_low'].to_numpy(), alt['ci_beta1_high'].to_numpy(), cfg.beta1)

            summary_rows.extend([
                {'scenario': scenario, 'scenario_label': SCENARIO_LABELS[scenario], 'n': n, 'model': model_label, 'parameter': 'beta0',
                 'mean_hat': m0, 'bias': b0, 'variance': v0, 'mse': mse0, 'coverage': cov0, 'size_h0': size_h0, 'power_h1': power_h1},
                {'scenario': scenario, 'scenario_label': SCENARIO_LABELS[scenario], 'n': n, 'model': model_label, 'parameter': 'beta1',
                 'mean_hat': m1, 'bias': b1, 'variance': v1, 'mse': mse1, 'coverage': cov1, 'size_h0': size_h0, 'power_h1': power_h1}
            ])

            temp = alt.copy()
            temp['scenario'] = scenario
            temp['scenario_label'] = SCENARIO_LABELS[scenario]
            temp['n'] = n
            temp['model'] = model_label
            estimation_rows.extend(temp.to_dict(orient='records'))

            if n == representative_n:
                slug = f"{SCENARIO_LABELS[scenario]}_n{n}_{model_label}"
                save_diagnostics(FIGURES, slug, diagnostic, alt['beta1_hat'].to_numpy())

summary_df = pd.DataFrame(summary_rows).sort_values(by=['scenario', 'n', 'model', 'parameter'])
est_df = pd.DataFrame(estimation_rows).sort_values(by=['scenario', 'n', 'model'])
compact_df = summary_df[['scenario_label', 'n', 'model', 'parameter', 'bias', 'variance', 'mse', 'coverage', 'size_h0', 'power_h1']]

summary_df.to_csv(TABLES / 'resumo_metricas.csv', index=False)
compact_df.to_csv(TABLES / 'resumo_compacto.csv', index=False)
est_df.to_csv(TABLES / 'estimativas_brutas.csv', index=False)

print('Concluido. Arquivos gerados em:', OUTPUTS)

Concluido. Arquivos gerados em: e:\MYAREA\AREA_DEV\Faculdade\ModelosRegressaoI\EstudoDeSimulacao\outputs


In [10]:
compact_df.head(20)

,scenario_label,n,model,parameter,bias,variance,mse,coverage,size_h0,power_h1
20,C5_erros_correlacionados_ar1,30,original_scale,beta0,-0.003424,0.174261,0.174098,0.561,0.044,1.0
21,C5_erros_correlacionados_ar1,30,original_scale,beta1,0.000569,0.023720,0.023697,0.940,0.044,1.0
22,C5_erros_correlacionados_ar1,100,original_scale,beta0,-0.007836,0.056076,0.056081,0.594,0.055,1.0
23,C5_erros_correlacionados_ar1,100,original_scale,beta1,-0.000749,0.007394,0.007387,0.958,0.055,1.0
0,C1_classico_normal,30,original_scale,beta0,-0.001837,0.035229,0.035198,0.948,0.052,1.0
1,C1_classico_normal,30,original_scale,beta1,0.004187,0.026348,0.026339,0.954,0.052,1.0
2,C1_classico_normal,100,original_scale,beta0,0.003529,0.009418,0.009421,0.953,0.046,1.0
3,C1_classico_normal,100,original_scale,beta1,-0.003099,0.007404,0.007406,0.951,0.046,1.0
26,C6_relacao_exponencial_lognormal,30,log_transform,beta0,-0.002164,0.005569,0.005568,0.945,0.058,1.0
27,C6_relacao_exponencial_lognormal,30,log_transform,beta1,0.000425,0.004226,0.004222,0.947,0.058,1.0


## Interpretacao resumida dos resultados

- **C1 (classico normal):** vies muito proximo de zero e cobertura proxima de 95%.
- **C2 e C3:** vies baixo com pequenas alteracoes de variabilidade/cobertura.
- **C4a (media nao nula constante):** forte vies no intercepto.
- **C4b (erro dependente de X):** vies relevante em `beta1` e queda de cobertura.
- **C5 (AR1):** impacto inferencial, principalmente no intercepto em amostra menor.
- **C6:** ajuste na escala original fica mal especificado; o ajuste em `log(Y)` recupera bom desempenho.

Conclusao geral: o MRLS e robusto em alguns cenarios de nao normalidade, mas sensivel a violacoes de exogeneidade e a especificacao funcional incorreta.

## Metodologia top-down (resumo)

1. Definicao do objetivo cientifico e dos cenarios.
2. Desenho experimental (parametros, tamanhos amostrais, repeticoes, semente).
3. Implementacao modular (geracao, ajuste, metricas, diagnosticos).
4. Execucao completa da simulacao e consolidacao em tabelas.
5. Analise comparativa e interpretacao por cenario.

Este notebook foi estruturado para reproducao direta no Google Colab.